In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [4]:
def glass_brain_plot(df, title, cmap='Reds', vmax=None, display_mode='lyrz'):
    from nilearn import image
    from matplotlib.colors import Normalize
    import matplotlib.gridspec as gridspec
    # Load the 3D labeled atlas NIfTI
    label_img = image.load_img('data/wlabel_sPR06786_AD151295-0012-00001-000176-01_MT_ants.nii') # old : data/labels_neuromorphics_extra.nii (we don't know the patient) or data/wlabel_sPR06786_AD151295-0012-00001-000176-01_MT.nii (from ferath)
    label_data = label_img.get_fdata()

    # Make an empty stat map
    stat_data = np.zeros_like(label_data, dtype=float)

    # Fill in stat map based on region values
    for region_id, value in zip(df['region_idx'], df['value']):
        stat_data[label_data == region_id] = value

    #stat_data[np.abs(stat_data) < 1e-4] = 0
    #print(np.unique(stat_data, return_counts=True))
    # Create a new NIfTI image
    stat_img = image.new_img_like(label_img, stat_data)
    if not vmax:
        return plotting.plot_glass_brain(
        stat_img,
        display_mode=display_mode,  # Show all orthogonal projections
        colorbar=True,
        cmap=cmap,
        #threshold=1e-6,
        #vmin=vmin,
        #vmax=vmax,
        title=title,
        plot_abs=True
    )
    else:
        display = plotting.plot_glass_brain(
            stat_img,
            display_mode=display_mode,
            colorbar=False,
            cmap=cmap,
            cbar_tick_format='%i',
            vmin=0,
            vmax=vmax,
            threshold=0,
            title=title,
            plot_abs=True
        )
    # Shrink main plot area to leave space for colorbar
    fig = plt.gcf()
    # Manually resize each axes object to free space on the right
    for ax in fig.axes:
        pos = ax.get_position()
        ax.set_position([pos.x0, pos.y0, pos.width * 0.85, pos.height])  # shrink width

    # Create colorbar manually in reserved space
    # Add a new axis for the horizontal colorbar at the bottom
    cax = fig.add_axes([0.35, 0.08, 0.3, 0.02])  # [left, bottom, width, height]

    norm = Normalize(vmin=0, vmax=vmax)
    sm = cm.ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])

    cbar = plt.colorbar(sm, cax=cax, orientation='horizontal')
    cbar.set_ticks(np.arange(0, vmax + 1, 1))


    return display  


In [ ]:
average_stiffness_df = pd.DataFrame({
                'value' : region_averages.values(),
                'region_idx' : region_averages.keys()
            })
_fig = glass_brain_plot(df=average_stiffness_df, title=f"Average regional stiffness", display_mode='lyrz')
#_fig.savefig(f"figs/glm_LI/glasser/count_mod/{_beta}_glass_brain.png")
plt.show()